<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2016%20-%202026-05-14%20-%20Buenas%20pr%C3%A1cticas%20y%20portafolio/clase_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 16 · Buenas prácticas, testing y portafolio

Hasta aquí el código que escribiste ha sido una jam session. Notebooks que crecen, decisiones tomadas rápido, poca documentación. Eso es normal durante el desarrollo. Pero hoy convertimos eso en algo que sobrevive.

No por vanidad. Por sobrevivencia profesional. Dentro de 3 meses vuelves a este código y necesitas entenderlo sin odiar al Javier del pasado que lo escribió. Y un reclutador lo va a leer el mes que viene. Mejor que diga "este código es profesional" que "esto parece un prototipo".

Hoy: pruebas, estructura, principios de POO, cómo se ve un repositorio que impresiona.

> **Hoy haces** · Refactorizar el repo: estructura modular, al menos 5 tests con pytest que pasen, README final del proyecto. ~2h30.
>
> **Entrega** · Repo final reorganizado con `tests/`, `src/` y `README.md` completo, antes de la próxima clase.

In [ ]:
# --- Setup del entorno
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)

print("Setup completo ✓")

## POO: cuatro ideas

La **Programación Orientada a Objetos** organiza código alrededor de **objetos** que tienen **estado** (atributos) y **comportamiento** (métodos). Se asienta en 4 conceptos:

1. **Encapsulamiento**: agrupar datos y operaciones sobre esos datos en una unidad.
2. **Herencia**: una clase extiende otra, heredando y especializando comportamiento.
3. **Polimorfismo**: distintas clases responden al mismo mensaje diferente.
4. **Abstracción**: exponer solo lo esencial, ocultar la implementación.

En proyectos de datos **no abuses de POO**. Normalmente:
- Módulos (`src/io.py`, `src/features.py`) contienen funciones.
- Las clases aparecen para **pipelines de datos**, **modelos custom** o **transformadores**.

## SOLID: cinco principios para diseño limpio

Robert C. Martin (Uncle Bob) sistematizó 5 principios:

- **S** — Single Responsibility: una clase, una razón para cambiar.
- **O** — Open/Closed: abierto a extensión, cerrado a modificación.
- **L** — Liskov Substitution: una subclase puede reemplazar a su padre sin romper nada.
- **I** — Interface Segregation: muchas interfaces pequeñas mejor que una gigante.
- **D** — Dependency Inversion: depender de abstracciones, no de implementaciones.

No es dogma. Pero ayudan a detectar cuando el diseño se deteriora. La mayoría de problemas en codebases viejas violan al menos tres de estos.

## Ejemplo: DataPipeline

Una clase que encapsula carga, limpieza y transformación. Útil para que distintos experimentos reutilicen la misma lógica:

In [ ]:
class DataPipeline:
    """Pipeline de datos: carga, limpia, transforma."""
    
    def __init__(self, url: str, seed: int = 42):
        """Inicializar pipeline con URL y seed para reproducibilidad."""
        self.url = url
        self.seed = seed
        self.df = None
        self.X = None
        self.y = None
    
    def cargar(self) -> 'DataPipeline':
        """Cargar datos desde URL."""
        self.df = pd.read_csv(self.url)
        return self  # Retornar self para encadenar
    
    def limpiar(self, subset: list = None) -> 'DataPipeline':
        """Remover filas con nulos en columnas específicas."""
        if subset:
            self.df = self.df.dropna(subset=subset)
        else:
            self.df = self.df.dropna()
        return self
    
    def preparar(self, features: list, target: str) -> 'DataPipeline':
        """Separar X (features) e y (target)."""
        self.X = self.df[features]
        self.y = self.df[target]
        return self
    
    def info(self) -> dict:
        """Retornar info del pipeline."""
        return {
            "shape": self.df.shape if self.df is not None else None,
            "features": list(self.X.columns) if self.X is not None else None,
            "target": self.y.name if self.y is not None else None,
        }

# Uso
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
pipeline = (DataPipeline(url)
            .cargar()
            .limpiar(subset=["Age"])
            .preparar(["Age", "Pclass", "Fare"], "Survived"))

print("Pipeline:", pipeline.info())

¿Por qué una clase? Porque encapsula lógica reutilizable. Mañana necesitas otra pipeline con pasos distintos — heredas de `DataPipeline` y especializas. Sin duplicar.

Nota el encadenamiento: cada método retorna `self`, permitiendo `pipeline.cargar().limpiar().preparar()`. Es más legible que tres líneas separadas.

## Testing: escudo defensivo

Un **test** es código que ejecuta otro código y verifica que se comporte como esperamos. Tiene tres funciones:

1. **Documentación ejecutable**: los tests muestran cómo se usa el código.
2. **Red de seguridad**: refactorizar sin miedo a romper cosas.
3. **Presión de diseño**: código testeable es código bien diseñado.

La pirámide de tests (Mike Cohn):

```
       /\
      /  \   ← Tests de integración (pocos, lentos)
     /----\
    /      \ ← Tests unitarios (muchos, rápidos)
   /________\
```

pytest es el estándar en Python. Convención: funciones que empiezan por `test_` dentro de archivos que empiezan por `test_`.

## Tests en acción

In [ ]:
# Guardar como tests/test_pipeline.py
TEST_CODE = '''
import pytest
from src.pipeline import DataPipeline

URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

def test_cargar():
    """Pipeline carga datos correctamente."""
    p = DataPipeline(URL).cargar()
    assert p.df is not None
    assert p.df.shape[0] > 0

def test_limpiar():
    """Limpiar remueve nulos."""
    p = DataPipeline(URL).cargar()
    antes = p.df.shape[0]
    p.limpiar(subset=["Age"])
    despues = p.df.shape[0]
    assert despues < antes  # Debe remover al menos 1 fila

def test_preparar():
    """Preparar separa X e y."""
    p = (DataPipeline(URL)
         .cargar()
         .limpiar(subset=["Age"])
         .preparar(["Age", "Pclass"], "Survived"))
    
    assert p.X is not None
    assert p.y is not None
    assert p.X.shape[0] == p.y.shape[0]
    assert p.X.shape[1] == 2  # 2 features
'''
print(TEST_CODE)
print("\nEjecución: pytest -v tests/test_pipeline.py")

Cuando corre `pytest -v`, verás algo como:

```
tests/test_pipeline.py::test_cargar PASSED
tests/test_pipeline.py::test_limpiar PASSED
tests/test_pipeline.py::test_preparar PASSED
```

Si una condición falla (un `assert` retorna False), el test falla. Y sabes dónde.

## PEP 8: estilo consistente

**PEP 8** es la guía oficial de estilo de Python. Puntos clave:

- **Indentación**: 4 espacios, no tabs.
- **Nombres**: `snake_case` variables/funciones, `PascalCase` clases, `MAYUSCULAS` constantes.
- **Línea máx**: 79 caracteres (o 99 si el equipo acuerda).
- **Espacios**: `a = b + c`, no `a=b+c`.
- **Imports**: primero estándar, luego terceros, luego local. Cada import una línea.

Herramientas para aplicarlo automáticamente:
- **`black`**: formatea el código (opinionado, sin configuración).
- **`ruff`** o **`flake8`**: detectan violaciones.
- **`isort`**: ordena imports.

Un `pre-commit hook` las corre antes de cada commit y evita que mandes código desordenado.

## Git para equipos

- **Branch por feature**: `git checkout -b feature/eda-semana-2`.
- **Commits pequeños con mensaje claro**: "Agrega limpieza de nulos en edad", no "cambios".
- **Pull requests**: otro del equipo revisa antes de mergear.
- **No subir datos crudos**: ya está en `.gitignore`.
- **Tag de versión final**: `git tag v1.0.0 -m "Entrega final"` para marcar hitos.

## Checkpoint: el repo que impresiona

Un repositorio que un reclutador ve y piensa "esta persona sabe entregar software":

- ✅ **README** claro: problema, datos, resultados clave, gráficos, cómo reproducir.
- ✅ **Licencia** (MIT es común).
- ✅ **requirements.txt** con versiones congeladas (`pip freeze > requirements.txt`).
- ✅ **.gitignore** serio (no `.venv`, `.DS_Store`, `data/raw/*`).
- ✅ **Notebooks numerados** (01_eda.ipynb, 02_modelo.ipynb) y narrados.
- ✅ **Carpeta src/** con módulos reutilizables.
- ✅ **Tests** ejecutables y pasando.
- ✅ **App/** con dashboard/API.
- ✅ **Capturas de pantalla** del resultado en el README.
- ✅ **Demo desplegada** (Streamlit Cloud, Hugging Face Spaces) con link en README.

In [ ]:
# Estructura recomendada
estructura = '''
mi_proyecto/
├── README.md                    # Tarjeta de presentación
├── LICENSE                      # MIT o similar
├── requirements.txt             # pip freeze
├── .gitignore                   # Ignorar .venv, datos, etc
├── setup.py                     # (opcional) para instalar como paquete
│
├── notebooks/
│   ├── 01_eda.ipynb
│   ├── 02_modelo.ipynb
│   └── 03_validacion.ipynb
│
├── src/
│   ├── __init__.py
│   ├── data.py                  # Carga, limpieza
│   ├── features.py              # Transformaciones
│   ├── model.py                 # Lógica del modelo
│   └── utils.py                 # Utilidades
│
├── tests/
│   ├── test_data.py
│   ├── test_features.py
│   └── test_model.py
│
├── app/
│   └── app.py                   # Streamlit o FastAPI
│
├── data/
│   ├── raw/                     # Nunca en Git
│   └── processed/               # Nunca en Git
│
└── results/                     # Gráficos finales, reports
'''
print(estructura)

## Para tu equipo

Reestructuren el código del proyecto:

1. **Extraigan lógica** a un módulo en `src/` con al menos 2 clases o 5 funciones.
2. **Escriban 5 tests** con pytest. Ejecutables y pasando.
3. **Actualicen README** con: qué resuelven, cómo reproducir, qué aprendieron.
4. **Creen requirements.txt** con versiones congeladas.
5. **Hagan commit** con mensaje claro: "refactor: modularizar pipeline de datos".

El entregable es el repo estructurado con tests pasando.

> **Recordatorio** · Repo reorganizado con tests al día antes de la presentación.

---

Hoy pasaste de "código que funciona" a "código profesional". Tests. Estructura. Principios. Documentación.

No es porque los principios sean el objetivo. Es porque son el **medio** para que el código sobreviva en el mundo real — donde otros lo leerán, lo extenderán, lo críticarán, y esperarán que sea mantenible.

Mañana presentan. El trabajo técnico ya está. Ahora toca defenderlo.